This file contains my solution to a task from [Data Science Simulator](https://karpov.courses/simulator-ds), a hands-on training course in data analysis and machine learning by karpov.courses.

This file is shared with the permission of the course authors, and in a reduced form: the original problem statements and datasets are omitted. What's kept is my own code, reasoning, and commentary.

## Overview

In this task we work with the website of an affiliate CPA network. We play the role of a beginner affiliate marketer (arbitrageur) who has traffic (an audience) but doesn't know which offers will convert best for that audience.

Smart-Link is an ML service that, given a broad group of offers, automatically figures out which of them convert best (and/or bring in the most money) for a given stream of traffic. So instead of sending all traffic to a single offer, the marketer spreads it across several.

In this task I implement:
- A reinforcement learning problem (multi-armed bandits)
- A simulation of traffic arbitrage (the environment for the agent)
- Deployment of the ML service (FastAPI)

The bandit is solved step by step with the following strategies:
- Random offer selection
- Greedy algorithm
- Epsilon-greedy algorithm
- Thompson sampling in two flavours: known variance / unknown mean, and unknown variance / unknown mean

### Step 1

Let's start with the simplest model: for any incoming click we show a random banner. The model has two stages:
- candidate selection (the offers a user can physically go to and subscribe to, depending on their geo, device, source, etc.)
- offer selection (the part I implement).

The candidate model is already provided.

The ML service receives a click (click ID) together with a set of candidate offers (offer IDs). We want to pick a random one of them.

In [ ]:
import random

from fastapi import FastAPI
import uvicorn

app = FastAPI()

@app.get("/sample/")
def sample(offer_ids: str) -> dict[str: int]:
    "Sample random offer from the input"
    #Parse offers ids
    offers = [int(offer) for offer in offer_ids.split(",")]

    #Sample random offer ID
    offer_id = random.choice(offers)

    #Prepare response
    response = {
        "offer_id": offer_id,
    }

    return response

def main():
    """Run application"""
    uvicorn.run("app:app", host="localhost")

if __name__ == "__main__":
    main()

### Step 2

At this step I update the logic of `/sample` and add two new endpoints:
- a PUT request `/feedback`
- a GET request `/stats`

The overall scheme is as follows. `/sample` returns an `offer_id` that is shown to the user for their `click_id`. At this point the link between `click_id` and `offer_id` is stored. Through the PUT `/feedback` request the advertiser reports whether a conversion happened for that click. This lets me collect statistics per `offer_id`, which are returned by the GET `/stats` request:
- `clicks` — number of clicks on the offer
- `conversions` — how many times a conversion into an action happened
- `reward` — total reward earned on the offer by the webmasters
- `cr` — conversions-to-clicks ratio
- `rpc` — average revenue per click

**Updated `/sample` logic.**
For the first 100 clicks sent to the service a random offer is chosen (for initialization). After that we pick the one (among the candidate banners) that maximizes reward per click. In the multi-armed bandit setting this strategy is called greedy. Note that as statistics get updated the reward per click can change, so the "best" offer can change too. We choose only among the candidate banners passed in `offer_ids`. If there is no suitable offer, I return the first one. For example, if the input is `offer_ids: [45, 67]` but statistics for both are zero, we pick 45.

**Data storage**

I keep the data only while the app is running. For that I initialize global variables that are cleared on shutdown via lifespan.

Dictionaries used:
- `clicks_offers` for the one-to-one click -> offer mapping
- `offers_conversions` to count conversions per offer
- `offers_rewards` to count rewards per offer
- `offers_recommendations` to count how many times each offer was shown (recommended).

In [ ]:
from collections import defaultdict
import random
from contextlib import asynccontextmanager

from fastapi import FastAPI
import uvicorn

stats_clicks = {"counter_clicks": 0}
clicks_offers = {}
offers_conversions = defaultdict(int)
offers_rewards = defaultdict(float)
offers_recommendations = defaultdict(int)


@asynccontextmanager
async def lifespan(app: FastAPI) -> None:
    yield
    stats_clicks["counter_clicks"] = 0
    clicks_offers.clear()
    offers_conversions.clear()
    offers_rewards.clear()
    offers_recommendations.clear()

app = FastAPI(lifespan=lifespan)

def find_best_offer(offers: list) -> int:
    """Find an offer with the highest rpc"""
    best_rpc = -1
    best_offer = -1
    for offer in offers:
        rewards = offers_rewards[offer]
        views = offers_recommendations[offer]
        if rewards != 0 and views != 0:
            rpc = rewards/views
            if rpc > best_rpc:
                best_rpc = rpc
                best_offer = offer

    return best_offer

@app.get("/sample/")
def sample(click_id: int, offer_ids: str) -> dict:
    "Sample random offer from the input"
    #Parse offers ids
    offers = [int(offer) for offer in offer_ids.split(",")]

    sampler = "greedy"
    stats_clicks["counter_clicks"] += 1

    if stats_clicks["counter_clicks"] <= 100:
        offer_id = random.choice(offers)
        sampler = "random"
    else:
        offer_id = find_best_offer(offers) if find_best_offer(offers) != -1 else offers[0]

    clicks_offers[click_id] = offer_id
    offers_recommendations[offer_id] += 1


    #Prepare response
    response = {
        "click_id": click_id,
        "offer_id": offer_id,
        "sampler": sampler,
    }

    return response

@app.put("/feedback/")
def feedback(click_id: int, reward: float) -> dict:
    """Get feedback for particular click"""
    # Response body consists of click ID
    # and accepted click status (True/False)
    offer_id = clicks_offers.get(click_id, None)
    response = {
        "click_id": click_id,
        "offer_id": offer_id,
        "is_conversion": True if reward != 0 else False,
        "reward": reward
    }

    #Update offer stats in case there was conversion and offer_id exists
    if response["is_conversion"] and offer_id:
        offers_conversions[offer_id] += 1
        offers_rewards[offer_id] += reward


    return response

@app.get("/offer_ids/{offer_id}/stats/")
def stats(offer_id: int) -> dict:
    """Return statistics for the offer"""

    clicks = offers_recommendations[offer_id]
    conversions = offers_conversions[offer_id]
    reward = offers_rewards[offer_id]
    cr = conversions/clicks if clicks !=0 else 0
    rpc = reward/clicks if clicks !=0 else 0

    response = {
        "offer_id": offer_id,
        "clicks": clicks,
        "conversions": conversions,
        "reward": reward,
        "cr": cr,
        "rpc": rpc,
    }

    return response

def main():
    """Run application"""
    uvicorn.run("app:app", host="localhost")

if __name__ == "__main__":
    main()

### Step 3

At this step I change the logic of `sample`: instead of the greedy strategy I introduce the epsilon-greedy mechanic. Under this strategy, with probability 1-epsilon we pick the offer with the best reward per click (exploitation in bandit terms), and with probability epsilon we randomly pick among the remaining offers (exploration).

Only the `sample` function changes. The random selection for the first hundred clicks, as before, is no longer needed.

In [ ]:
from collections import defaultdict
import random
from contextlib import asynccontextmanager

from fastapi import FastAPI
import uvicorn

stats_clicks = {"counter_clicks": 0}
clicks_offers = {}
offers_conversions = defaultdict(int)
offers_rewards = defaultdict(float)
offers_recommendations = defaultdict(int)


@asynccontextmanager
async def lifespan(app: FastAPI) -> None:
    yield
    stats_clicks["counter_clicks"] = 0
    clicks_offers.clear()
    offers_conversions.clear()
    offers_rewards.clear()
    offers_recommendations.clear()

app = FastAPI(lifespan=lifespan)

def find_best_offer(offers: list) -> int:
    """Find an offer with the highest rpc"""
    best_rpc = -1
    best_offer = -1
    for offer in offers:
        rewards = offers_rewards[offer]
        views = offers_recommendations[offer]
        if rewards != 0 and views != 0:
            rpc = rewards/views
            if rpc > best_rpc:
                best_rpc = rpc
                best_offer = offer

    return best_offer

@app.get("/sample/")
def sample(click_id: int, offer_ids: str) -> dict:
    "Epsilon-greedy strategy"
    epsilon = 0.1
    sampler = "greedy"
    #Parse offers ids
    offers = [int(offer) for offer in offer_ids.split(",")]

    stats_clicks["counter_clicks"] += 1

    a = random.uniform(0, 1)
    if a > (1 - epsilon):
        offer_id = random.choice(offers)
        sampler = "random"
    else:
        offer_id = find_best_offer(offers) if find_best_offer(offers) != -1 else offers[0]

    clicks_offers[click_id] = offer_id
    offers_recommendations[offer_id] += 1

    #Prepare response
    response = {
        "click_id": click_id,
        "offer_id": offer_id,
        "sampler": sampler,
    }
    return response

@app.put("/feedback/")
def feedback(click_id: int, reward: float) -> dict:
    """Get feedback for particular click"""
    # Response body consists of click ID
    # and accepted click status (True/False)
    offer_id = clicks_offers.get(click_id, None)
    response = {
        "click_id": click_id,
        "offer_id": offer_id,
        "is_conversion": True if reward != 0 else False,
        "reward": reward
    }

    #Update offer stats in case there was conversion and offer_id exists
    if response["is_conversion"] and offer_id:
        offers_conversions[offer_id] += 1
        offers_rewards[offer_id] += reward


    return response

@app.get("/offer_ids/{offer_id}/stats/")
def stats(offer_id: int) -> dict:
    """Return statistics for the offer"""

    clicks = offers_recommendations[offer_id]
    conversions = offers_conversions[offer_id]
    reward = offers_rewards[offer_id]
    cr = conversions/clicks if clicks !=0 else 0
    rpc = reward/clicks if clicks !=0 else 0

    response = {
        "offer_id": offer_id,
        "clicks": clicks,
        "conversions": conversions,
        "reward": reward,
        "cr": cr,
        "rpc": rpc,
    }

    return response

def main():
    """Run application"""
    uvicorn.run("app:app", host="localhost")

if __name__ == "__main__":
    main()


### Step 4

I want to improve the selection with Thompson sampling in order to minimize the regret. Thompson sampling is an optimal algorithm in which the exploration-exploitation trade-off does not prevent us from reaching lower regret than the epsilon-greedy algorithm.

The basic version of Thompson sampling assumes a Bernoulli trial with answers 1 and 0. In that case we could use a Beta distribution and, updating its parameters as feedback arrives, train the model.

However, here we deal with real-valued rewards. So we need a Bayesian approach with conjugate priors.

I implemented two approaches:
- conjugate priors with known variance and unknown mean
- conjugate priors with unknown variance and unknown mean

Details of the algorithms can be found in [1].

#### 1. Known variance, unknown mean

Here the mean refers to the mean reward value — rpc. The idea: we assume we know the variance of the reward distribution. It isn't actually given in the task, but we can assume the scale of the rewards is known in advance. The mean, however, we can't know — that's what we approximate as we receive feedback from the advertisers.

In this case both the prior and the posterior distribution of rpc are normal. After each `/sample` request I sample the posterior distribution for each of the offered banners and pick the maximum of the returned values. Each banner has its own distribution parameters, updated as rewards arrive through `/feedback`.

I implemented the `GaussThompson` class with the following:
- `tau` — the inverse assumed variance, a fixed value
- hyperparameters `mu0`, `tau0` — parameters of the prior distribution, updated as new information arrives
- a `sample` method to sample the normal distribution
- an `update` method to update the posterior parameters

Details on how `mu0`, `tau0` change can be found in [1] and [2].

Tuning the parameters, I reached the best result with `v` = 1.0, `mu0` = 10.0, `v0` = 25. The biggest reduction in regret came from decreasing `v` from 25.0 to 1.0 — meaning the model moves quite quickly from the exploration stage to exploitation. With a large `v` I lost a lot in regret due to excessive exploration.

#### 2. Unknown variance and mean

A model in which we know nothing about the rewards.

Here the joint posterior over the mean and the precision is a Normal-Gamma distribution. The update logic is now: first we sample the precision (inverse variance) of rpc from the Gamma distribution (which in the previous case we assumed known), then we sample rpc from the normal distribution using the variance we just obtained. The Gamma distribution thus expresses our uncertainty about the variance.

I implemented the `GammaNormalThompson` class with the following:
- hyperparameters `alpha`, `beta` — parameters of the prior distribution, updated in the posterior as new information arrives; they characterize the degree of uncertainty about the variance
- hyperparameter `mu0` — the prior guess about the mean
- hyperparameter `nu` — how many observations the current mean guess is built on (before observations — about the prior mean; after — about the current mean we compute)
- a `sample` method to sample the Normal-Gamma distribution
- an `update` method to update the posterior parameters

In the end this method lost to `GaussThompson` on regret minimization. This can be interpreted as: adding one more source of uncertainty in the form of sampling the variance makes me spend too much time on exploration, which worsens the regret.

Materials:
[1] - https://towardsdatascience.com/thompson-sampling-using-conjugate-priors-e0a18348ea2d/
[2] - https://en.wikipedia.org/wiki/Conjugate_prior#When_likelihood_function_is_a_continuous_distribution

In [ ]:
from collections import defaultdict
from contextlib import asynccontextmanager

from fastapi import FastAPI
import uvicorn
import numpy as np

class GaussThompson:
    """Gaussian Thompson Sampling with known variance and unknown mean"""
    def __init__(self, v: float = 1.0, mu0: float = 10.0, v0: float = 25) -> None:
        self.tau = 1/v #Suggested precision (inverse variance)

        #Hyperparameters: average and precision (inverse variance) of the prior distribution
        #They will be updated during the process
        self.mu0 = mu0
        self.tau0 = 1/v0


        self.conversions = 0 #Number of converisions from this offer
        self.rewards = 0.0 #Number rewards this offer received
        self.n = 0 #Number of times this offer was suggested

    def sample(self) -> float:
        """Return sample from the normal distribution with mu0 and v0 parameter"""
        return np.random.normal(self.mu0, np.sqrt(1/self.tau0))

    def update(self, reward: float) -> None:
        """Update the aposterior distribution parameters"""

        if reward != 0:
            self.conversions += 1
        self.rewards += reward

        self.mu0 = (self.tau0 * self.mu0 + self.tau * reward) \
                    / (self.tau0 + self.tau)
        self.tau0 += self.tau

class GammaNormalThompson:
    """Gaussian Thompson Sampling with unknown variance and unknown mean"""
    def __init__(self, mu0: float = 10.0, nu: int = 1,
                alpha: float = 1.0, beta: float = 100.0) -> None:

        #Prior values that will be updated after each observation
        #Observation means receiving feedback and reward value within it
        self.mu = mu0 #Mean value
        self.nu = nu #Number of observations
        self.alpha = alpha #alpha parameter for Gamma distribution
        self.beta = beta #beta parameter for Gamma distribution

        self.conversions = 0 #Number of converisions from this offer
        self.rewards = 0.0 #Number rewards this offer received
        self.n = 0 #Number of times this offer was suggested

    def sample(self) -> float:
        """Return sample from the gamma-normal distribution"""

        #Sample gamma distribution and obtain precision
        sample_precision = np.random.gamma(self.alpha, 1/self.beta)
        sample_variance = 1/sample_precision if sample_precision != 0 else 1e6

        #Sample normal distribution and use obtained variance
        return np.random.normal(self.mu, np.sqrt(sample_variance))

    def update(self, reward: float) -> None:
        """Update the aposterior distribution parameters"""
        if reward != 0:
            self.conversions += 1
        self.rewards += reward

        self.beta += self.nu / (self.nu + 1) * ((reward - self.mu)**2)/2
        self.mu = (self.nu * self.mu + reward) / (self.nu + 1)

        self.nu += 1
        self.alpha += 1/2


clicks_offers = {}
offers = defaultdict(GammaNormalThompson)


@asynccontextmanager
async def lifespan(app: FastAPI) -> None:
    yield
    clicks_offers.clear()
    offers.clear()

app = FastAPI(lifespan=lifespan)

@app.get("/sample/")
def sample(click_id: int, offer_ids: str) -> dict:
    "Gaussian Thompson Sampling"
    #Parse offers ids
    candidates = [int(offer) for offer in offer_ids.split(",")]

    best_rpc = float("-inf")
    best_offer = -1

    for i in candidates:
        offer = offers[i]
        sample_rpc = offer.sample()
        if sample_rpc > best_rpc:
            best_rpc = sample_rpc
            best_offer = i

    offer_id = best_offer
    offers[offer_id].n += 1
    clicks_offers[click_id] = offer_id

    #Prepare response
    response = {
        "click_id": click_id,
        "offer_id": offer_id,
    }
    return response

@app.put("/feedback/")
def feedback(click_id: int, reward: float) -> dict:
    """Get feedback for particular click"""
    # Response body consists of click ID
    # and accepted click status (True/False)
    offer_id = clicks_offers.get(click_id, None)
    response = {
        "click_id": click_id,
        "offer_id": offer_id,
        "is_conversion": True if reward != 0 else False,
        "reward": reward
    }

    #Update offer stats
    if offer_id is not None:
        offer = offers[offer_id]
        offer.update(reward)

    return response

@app.get("/offer_ids/{offer_id}/stats/")
def stats(offer_id: int) -> dict:
    """Return statistics for the offer"""
    offer = offers[offer_id]

    clicks = offer.n
    conversions = offer.conversions
    reward = offer.rewards
    cr = conversions/clicks if clicks !=0 else 0
    rpc = reward/clicks if clicks !=0 else 0

    response = {
        "offer_id": offer_id,
        "clicks": clicks,
        "conversions": conversions,
        "reward": reward,
        "cr": cr,
        "rpc": rpc,
    }
    return response

def main():
    """Run application"""
    uvicorn.run("app:app", host="localhost")

if __name__ == "__main__":
    main()
